<a href="https://colab.research.google.com/github/limkc0116/FYP/blob/main/VGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = '/content/drive/MyDrive/FYPCNN/FYP-3-74/train'
BATCH_SIZE = 8
classes = ['Center_Console', 'Eat_or_Drink', 'Glove_Open', 'Hand_Off', 'Mirror_adj', 'Phone_Use']

# Training generator (with augmentation)
train_gen = ImageDataGenerator(
  rescale=1./255,
  validation_split=0.2,
  rotation_range=10,  # Reverted to 10
  width_shift_range=0.1, # Reverted to 0.1
  height_shift_range=0.1, # Reverted to 0.1
  shear_range=0.1, # Reverted to 0.1
  zoom_range=0.2,   # Reverted to 0.2
  # Removed brightness_range
  horizontal_flip=True,
  fill_mode='nearest'
)

# Validation generator (NO augmentation)
val_gen = ImageDataGenerator(
  rescale=1./255,
  validation_split=0.2
)

train_data = train_gen.flow_from_directory(
  train_dir,
  target_size=(224,224),
  batch_size=BATCH_SIZE,
  classes=classes,
  subset='training',
  seed=42
  # shuffle=True is the default here, which is perfectly fine for training!
)

val_data = val_gen.flow_from_directory(
    train_dir, # Corrected directory to train_dir as validation_split is used
    target_size=(224, 224),
    batch_size=BATCH_SIZE, # Using BATCH_SIZE for consistency
    classes=classes, # Specify classes for categorical output
    subset='validation',
    seed=42
)

Found 3076 images belonging to 6 classes.
Found 766 images belonging to 6 classes.


In [ ]:
import numpy as np
from sklearn.utils import class_weight
from tensorflow.keras.callbacks import EarlyStopping

labels = train_data.classes

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(labels),
    y=labels
)

class_weights = dict(enumerate(weights))
print("Calculated Class Weights:", class_weights)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

Calculated Class Weights: {0: np.float64(0.7365900383141762), 1: np.float64(0.7732528908999498), 2: np.float64(1.6020833333333333), 3: np.float64(4.101333333333334), 4: np.float64(1.1144927536231883), 5: np.float64(0.6313628899835796)}


In [ ]:
# Keep ONLY this, delete Cell 4 entirely
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,              # ✅ Will stop early when loss stops improving
    restore_best_weights=True
)

In [ ]:
from tensorflow.keras.applications import VGG19
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping # Make sure EarlyStopping is imported

LEARNING_RATE = 0.01  # Initial learning rate for frozen layers

base_vgg19 = VGG19(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_vgg19.trainable = False  # Freeze all layers first

model_vgg = models.Sequential([
    base_vgg19,
    layers.Flatten(),
    layers.Dense(256),
    layers.BatchNormalization(), # Re-added Batch Normalization for stability
    layers.Activation('relu'),   # Activation after BN
    layers.Dropout(0.3),
    layers.Dense(len(classes), activation='softmax')
])

model_vgg.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/FYPCNN/FYP-3-74/vgg19_best.keras',
    monitor='val_accuracy',
    save_best_only=True
)

early_stop = EarlyStopping( # Re-defining here to ensure it's available
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

print("Phase 1: Training frozen base...")
history_vgg19 = model_vgg.fit(
    train_data,
    validation_data=val_data,
    epochs=100,
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# --- Phase 2: Fine-tuning (Removed as per user request to avoid 100% accuracy) ---
# Unfreeze the last 4 convolutional blocks (VGG19 has 5 blocks)
# block5_conv1, block5_conv2, block5_conv3, block5_pool
# block4_conv1, block4_conv2, block4_conv3, block4_pool
# This loop will unfreeze layers starting from the specified index till the end.
# VGG19 has 19 layers in its base. Unfreezing [-12:] would be block4 and block5

# print("\nPhase 2: Fine-tuning - Unfreezing and training more layers...")

# # Unfreeze the last 4 layers of the VGG19 base model (last conv block)
# # VGG19 has 19 layers in its base. [-4:] corresponds to the last conv block
# for layer in base_vgg19.layers[-4:]:
#     layer.trainable = True

# # It is crucial to recompile the model after unfreezing layers
# model_vgg.compile(
#     optimizer=Adam(learning_rate=1e-5),  # Use a VERY low learning rate for fine-tuning
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# # Continue training with fine-tuned layers
# history_finetune = model_vgg.fit(
#     train_data,
#     validation_data=val_data,
#     epochs=50, # You can adjust this, but early stopping will control it
#     class_weight=class_weights,
#     callbacks=[early_stop, checkpoint], # Use the same early stopping and checkpoint
#     verbose=1
# )

# Save the final model (after Phase 1 only)
model_vgg.save('/content/drive/MyDrive/FYPCNN/FYP-3-74/vgg19_frozen_base_trained.keras')

Phase 1: Training frozen base...
Epoch 1/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 74s 182ms/step - accuracy: 0.7152 - loss: 0.6750 - val_accuracy: 0.9164 - val_loss: 0.1473
Epoch 2/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 69s 178ms/step - accuracy: 0.8615 - loss: 0.3146 - val_accuracy: 0.9465 - val_loss: 0.1702
Epoch 3/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 79s 172ms/step - accuracy: 0.8859 - loss: 0.2787 - val_accuracy: 0.9883 - val_loss: 0.0800
Epoch 4/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 68s 178ms/step - accuracy: 0.8911 - loss: 0.2700 - val_accuracy: 0.8551 - val_loss: 0.6560
Epoch 5/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 65s 169ms/step - accuracy: 0.9008 - loss: 0.2452 - val_accuracy: 0.9830 - val_loss: 0.0782
Epoch 6/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 64s 167ms/step - accuracy: 0.9142 - loss: 0.2105 - val_accuracy: 0.9687 - val_loss: 0.1090
Epoch 7/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 66s 172ms/step - accuracy: 0.9275 - loss: 0.1941 - val_accuracy: 0.9765 - val_loss: 0.0756
Epoch 8/100
385/385 ━━━━━━━━━━━━━━━━━━━━ 65

In [ ]:
import os

train_files = set()
for cls in classes:
    folder = f'/content/drive/MyDrive/FYPCNN/FYP-3-74/train/{cls}'
    train_files.update(os.listdir(folder))

test_files = set()
for cls in classes:
    folder = f'/content/drive/MyDrive/FYPCNN/FYP-3-74/test/{cls}'
    test_files.update(os.listdir(folder))

overlap = train_files & test_files
print(f"Overlapping files: {len(overlap)}")  # Should be 0

Overlapping files: 0


In [ ]:
for cls in classes:
    train_path = f'/content/drive/MyDrive/FYPCNN/FYP-3-74/train/{cls}'
    test_path = f'/content/drive/MyDrive/FYPCNN/FYP-3-74/test/{cls}'
    print(f"{cls}: train={len(os.listdir(train_path))}, test={len(os.listdir(test_path))}")

Center_Console: train=870, test=62
Eat_or_Drink: train=828, test=54
Glove_Open: train=400, test=33
Hand_Off: train=156, test=11
Mirror_adj: train=574, test=40
Phone_Use: train=1014, test=73


In [ ]:
from sklearn.metrics import classification_report
import numpy as n

val_data.reset()
y_true = val_data.classes
target_names = ['DANGER', 'MANUAL', 'VISUAL']
print("Evaluating VGG19...")
pred_vgg19 = model_vgg.predict(val_data)
print(classification_report(y_true, np.argmax(pred_vgg19, axis=1), target_names=target_names))
print("\nEvaluating VGG16...")
pred_vgg16 = model_vgg16.predict(val_data)
print(classification_report(y_true, np.argmax(pred_vgg16, axis=1), target_names=target_names))

Evaluating VGG19...
42/42 ━━━━━━━━━━━━━━━━━━━━ 5s 108ms/step
              precision    recall  f1-score   support

      DANGER       0.13      0.13      0.13        15
      MANUAL       0.27      0.27      0.27        64
      VISUAL       0.79      0.79      0.79       252

    accuracy                           0.66       331
   macro avg       0.39      0.39      0.39       331
weighted avg       0.66      0.66      0.66       331


Evaluating VGG16...
42/42 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step
              precision    recall  f1-score   support

      DANGER       0.07      0.07      0.07        15
      MANUAL       0.23      0.23      0.23        64
      VISUAL       0.79      0.79      0.79       252

    accuracy                           0.65       331
   macro avg       0.36      0.36      0.36       331
weighted avg       0.65      0.65      0.65       331

